In [ ]:
# read all json files in the episodes_folder
import os
import json
import plotly.express as px
import plotly.subplots as sp
import pandas as pd
import numpy as np

episodes_folder = "../episodes"

scenes = ["grCommercial", "grHome", "nv", "vc"]
all_episodes = []
all_targets_df = pd.DataFrame()
dtg_df = pd.DataFrame()

def update_fig_style(fig, width=2000, height=500):
    fig.update_layout(font=dict(family="Times New Roman"))
    fig.update_layout(title_font=dict(size=24, family="Times New Roman"))
    fig.update_layout(xaxis_title_font=dict(size=20, family="Times New Roman"))
    fig.update_layout(yaxis_title_font=dict(size=20, family="Times New Roman"))
    fig.update_layout(legend_font=dict(size=20, family="Times New Roman"))
    fig.update_layout(font=dict(size=20))
    fig.update_layout(plot_bgcolor='rgba(0,0,0,0)')
    fig.update_layout(paper_bgcolor='rgba(0,0,0,0)')
    fig.update_layout(width=width, height=height)

for scene in scenes:
    json_files = [f for f in os.listdir(episodes_folder) if f.endswith('.json') if f.startswith(scene)]

    episodes = []
    for file in json_files:
        if file=="test_generator.json":
            continue
        with open(os.path.join(episodes_folder, file), 'r') as f:
            data = json.load(f)
            all_episodes.extend(data)
            episodes.extend(data)

    targets = [x["instruction"] for x in episodes]
    unique_targets = list(set(targets))
    print(f"Unique targets in {scene}: {len(unique_targets)}")
    print(unique_targets)
    targets_df = pd.DataFrame({"target": targets, "scene": scene}).sort_values(by="target", key=lambda x: x.map(targets.count), ascending=False)
    all_targets_df = pd.concat([all_targets_df, targets_df])

    dtg = pd.DataFrame()
    # distance from start position to closest goal
    for episode in episodes:
        start_position = episode["start_position"]
        goal_positions = [x["location"] for x in episode["goals"]]
        distances = [np.linalg.norm(np.array(start_position) - np.array(goal_position)) for goal_position in goal_positions]
        # add min distance to dtg
        dtg = pd.concat([dtg, pd.DataFrame({"distance": [min(distances)], "scene": [scene]})])
    dtg_df = pd.concat([dtg_df, dtg])

print(f"Total episodes: {len(all_episodes)}")
all_targets = [x["instruction"] for x in all_episodes]
unique_targets = list(set(all_targets))
print(f"Unique targets: {len(unique_targets)}")
print(unique_targets)


# plot target distribution
# fig = px.histogram(all_targets_df, x="target", color="scene")
# fig.update_xaxes(showticklabels=True, tickangle=45, automargin=True, row=1, col=i+1)
# update_fig_style(fig, 1500, 500)
# fig.write_image("figs/target_distribution.png")
# fig.show()

# plot distance to goal
dtg_df["distance"] = dtg_df["distance"].clip(upper=100)
# change color palette
fig = px.histogram(dtg_df, x="distance", color="scene", color_discrete_sequence=px.colors.qualitative.Vivid)
update_fig_style(fig, 900, 600)
fig.write_image("figs/distance_to_goal.png")
fig.show()


In [ ]:
dtg_df[dtg_df["scene"] == "grCommercial"]

In [ ]:
episodes[0]